# PSO-based DAMASK Crystal Plasticity Calibration

DAMASK version is DAMASK 3.0.0a7 in this code. This notebook calibrates crystal-plasticity parameters using a particle swarm optimizer (PSO) to iterated DAMASK simulations. The experimental stress-strain curve is read from the first two columns of `Stress_strain.csv` (`Strain`, `Stress`). Every new simulated curve (useing optimized parameters) is appended to the same CSV as paired columns named `epsilon_N` and `sigma_N`, where `N` is the global simulation counter.


## 1. Imports

Load the DAMASK, numerical libraries, interpolation tools, and the PySwarms optimizer used for the calibration loop.


In [1]:
#DAMASK version is DAMASK 3.0.0a7 in this code.
import damask
import h5py
import sys
import numpy as np
import csv
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from numpy import array
from numpy.linalg import norm
# PySwarms provides the particle swarm optimizer used to search the constitutive parameters.
import pyswarms as ps

## 2. Optimization Logs and Interpolation Function

Create fresh CSV logs for the parameters and errors, and define a function that converts any stress-strain column pair into a continuous interpolation function.


In [2]:
# -----------------------------------------------------------------------------
# Initialize optimization log files.
# -----------------------------------------------------------------------------
# Param.csv stores the constitutive parameters tested by each DAMASK simulation.
# Error.csv stores the corresponding relative L2 error for each simulated curve.
with open('Param.csv', 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['a_sl', 'h_0_sl_sl','xi_inf_sl','xi_0_sl'])

with open('Error.csv', 'w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['error'])

# -----------------------------------------------------------------------------
# Function: convert two CSV columns into an interpolation function.
# -----------------------------------------------------------------------------
# The first two columns of Stress_strain.csv contain the experimental curve:
#   Strain, Stress
# During optimization, each new DAMASK result is appended as another pair:
#   epsilon_1, sigma_1, epsilon_2, sigma_2, ...
def interpolate_data(filename, column_1, column_2):
    epsilon = pd.read_csv(filename, usecols=column_1)
    epsilon = np.asarray(epsilon).squeeze()
    epsilon = epsilon[~np.isnan(epsilon)]

    sigma = pd.read_csv(filename, usecols=column_2)
    sigma = np.asarray(sigma).squeeze()
    sigma = sigma[~np.isnan(sigma)]

    # Return stress as a function of strain, using linear interpolation between
    # the tabulated points in the selected columns.
    f = interp1d(epsilon, sigma, kind='linear')
    return f


## 3. Geometry and Material Setup

Read the RVE from the DREAM.3D file, save the DAMASK geometry, load phase definitions from YAML, and prepare the material configuration that will be modified during PSO.


In [3]:
# -----------------------------------------------------------------------------
# Build the DAMASK geometry and material configuration from the DREAM.3D file.
# -----------------------------------------------------------------------------
# The DREAM.3D file contains the real RVE microstructure and phase assignment.
filename_dream3d = 'RealRVE_1.dream3d'
g = damask.Grid.load_DREAM3D(filename_dream3d)

# Cut the simulation grid to a 50 x 50 x 1 canvas. This keeps the
# optimization affordable because every PSO particle requires a full DAMASK run.
g = g.canvas([50, 50, 1])
# DAMASK_grid reads geometry.vti during each simulation.
g.save('geometry')

# Load the material table from DREAM.3D, then replace the phase definitions with
# the local YAML files. Ferrite parameters are changed by the PSO later; Carbide
# parameters remain fixed unless you edit material_phase_Carbide.yaml.
config_material = damask.ConfigMaterial.load_DREAM3D(filename_dream3d)
config_material['phase']['Ferrite'] = damask.ConfigMaterial.load('material_phase_Ferrite.yaml')
config_material['phase']['Carbide'] = damask.ConfigMaterial.load('material_phase_Carbide.yaml')

# Use direct homogenization with one constituent per material point and pass the
# mechanical response directly to the grid solver.
config_material['homogenization']['direct'] = {'N_constituents': 1, 'mechanical': {'type': 'pass'}}
config_material.save()


## 4. Phase Count Check

Count Ferrite and Carbide entries in the material table as a lightweight sanity check before starting expensive simulations.


In [4]:
# -----------------------------------------------------------------------------
# Count how many material entries belong to Ferrite and Carbide.
# -----------------------------------------------------------------------------
# This is a quick sanity check that the DREAM.3D-derived material table contains
# the expected phase distribution before starting many expensive simulations.
NumFerrite = 0
Material_quantity = 0

for material in config_material['material']:
    Material_quantity += 1

for i in range(Material_quantity):
    if config_material['material'][i]['constituents'][0]['phase'] == 'Ferrite':
       NumFerrite += 1

NumCarbide = Material_quantity - NumFerrite
print(f"NumberOfFerrite={NumFerrite}")
print(f"NumberOfCarbide={NumCarbide}")


NumberOfFerrite=331
NumberOfCarbide=258


## 5. Load Case

Define the uniaxial deformation-rate boundary condition and write `load.yaml`, which each DAMASK run uses during optimization.


In [5]:
# -----------------------------------------------------------------------------
# Define the DAMASK mechanical load case.
# -----------------------------------------------------------------------------
# DAMASK boundary conditions use 'x' as the unknown/free component. Where dot_F
# is prescribed, dot_P must be unknown, and where dot_F is unknown, dot_P must be
# prescribed. This helper creates the complementary dot_P table automatically.
def inversion(l, fill=0):
    return [inversion(i, fill) if isinstance(i, list) else
            fill if i == 'x' else 'x' for i in l]

# Use the spectral_basic mechanical solver for this grid simulation.
load_case = damask.Config(solver={'mechanical': 'spectral_basic'},
                          loadstep=[])

# Apply a uniaxial deformation-rate condition in the y direction. The x and z
# normal deformation components are left free ('x'), while shear components are
# fixed to zero. The same condition is used for both load steps below.
dot_F = [['x', 0, 0],
         [0, 0.0001, 0],
         [0, 0, 'x']]

# Short initial step
loadstep = {'boundary_conditions': {'mechanical': {'dot_F': dot_F,
                                                   'dot_P': inversion(dot_F)}},
            'discretization': {'t': 30, 'N': 20},
            'f_out': 1}
load_case['loadstep'].append(loadstep)

# Longer loading step: reaches the strain range used for fitting the
# experimental stress-strain data, approximately up to 0.2 strain.
dot_F = [['x', 0, 0],
         [0, 0.0001, 0],
         [0, 0, 'x']]

loadstep = {'boundary_conditions': {'mechanical': {'dot_F': dot_F,
                                                   'dot_P': inversion(dot_F)}},
            'discretization': {'t': 1900, 'N': 50},
            'f_out': 1}
load_case['loadstep'].append(loadstep)

# DAMASK_grid will read this load file during every PSO trial.
# load_case.save('tension.yaml')
load_case.save('load.yaml')


## 6. Objective Function

For each PSO particle, write the candidate Ferrite parameters into `material.yaml`, run DAMASK, append the simulated curve to `Stress_strain.csv`, and compute the relative L2 error against the experimental curve.


In [8]:
# -----------------------------------------------------------------------------
# Define the PSO objective function.
# -----------------------------------------------------------------------------
count = 0
n_particles = 20       # Number of parameter sets evaluated per PSO iteration.
dimensions = 4         # h_0_sl-sl, a_sl, xi_inf_sl, and xi_0_sl are optimized.

# Error is evaluated only over this strain interval. 
optimization_range = np.arange(0.0035, 0.2, 0.0001)

# Interpolate the experimental curve from the first two columns of Stress_strain.csv.
# The experimental stress values are converted from MPa to Pa to match DAMASK output.
f_e = interpolate_data('Stress_strain.csv', ['Strain'], ['Stress'])
sigma_e = f_e(optimization_range) * 1e6

# PySwarms calls this function with I shaped as (n_particles, dimensions).
# The function must return one scalar cost for each particle in the current swarm.
def cost_function(I):
    global count  # Counts the total number of DAMASK simulations across all iterations.
    error1 = []   # Stores the objective values for this PSO population only.

    # Parameter order in the optimizer vector:
    #   I[:, 0] -> h_0_sl-sl  : initial hardening modulus for slip-slip interaction [Pa]
    #   I[:, 1] -> a_sl       : hardening exponent [-]
    #   I[:, 2] -> xi_inf_sl  : saturation slip resistance [Pa]
    #   I[:, 3] -> xi_0_sl    : initial slip resistance [Pa]
    h_0 = I[:, 0].tolist()
    a_sl = I[:, 1].tolist()
    xi_inf_sl = I[:, 2].tolist()
    xi_0_sl = I[:, 3].tolist()

    # Evaluate every particle in the swarm by writing its parameters to the
    # Ferrite phase, running DAMASK, post-processing the curve, and computing error.
    for i in range(0, n_particles):
        count = count + 1

        # Log the exact parameter set used for this simulation. The row number in
        # Param.csv corresponds to the same simulation number used in Error.csv and
        # the epsilon_N/sigma_N columns appended to Stress_strain.csv.
        with open('Param.csv', 'a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([a_sl[i], h_0[i], xi_inf_sl[i], xi_0_sl[i]])

        # ---------------------------------------------------------------------
        # Update Ferrite plasticity parameters in material.yaml.
        # ---------------------------------------------------------------------
        # Only Ferrite is optimized here. Carbide remains fixed from its YAML file.
        config_material['phase']['Ferrite']['mechanical']['plastic']['h_0_sl-sl'] = int(h_0[i])
        config_material['phase']['Ferrite']['mechanical']['plastic']['a_sl'] = round(a_sl[i], 2)
        config_material['phase']['Ferrite']['mechanical']['plastic']['xi_inf_sl'] = [int(xi_inf_sl[i])]
        config_material['phase']['Ferrite']['mechanical']['plastic']['xi_0_sl'] = [int(xi_0_sl[i])]
        config_material.save()

        # ---------------------------------------------------------------------
        # Run DAMASK for the current parameter set.
        # ---------------------------------------------------------------------
        # This Jupyter shell command reads load.yaml, geometry.vti, and material.yaml,
        # then writes geometry_load.hdf5. The tail command keeps notebook output short.
        # OMP_NUM_THREADS={8}
        ! mpiexec -n 1 DAMASK_grid --load load.yaml --geom geometry.vti | tail -n 2

        # ---------------------------------------------------------------------
        # Post-process the DAMASK result into average von Mises stress and strain.
        # ---------------------------------------------------------------------
        results = damask.Result('geometry_load.hdf5')
        results.add_stress_Cauchy()
        results.add_strain()
        results.add_strain('F_p')
        results.add_equivalent_Mises('sigma')
        results.add_equivalent_Mises('epsilon_V^0.0(F)')

        # Average all grid-point values at each output increment to obtain one
        # representative RVE stress-strain curve for this particle.
        sigma = [np.average(s) for s in results.place('sigma_vM').values()]
        epsilon = [np.average(e) for e in results.place('epsilon_V^0.0(F)_vM').values()]

        # ---------------------------------------------------------------------
        # Append the simulated curve to Stress_strain.csv.
        # ---------------------------------------------------------------------
        # The first two columns stay as the experimental curve. Every simulation
        # adds one new strain/stress pair: epsilon_1/sigma_1, epsilon_2/sigma_2, ...
        data_csv = pd.read_csv('Stress_strain.csv')
        data_csv[f'epsilon_{count}'] = pd.Series(epsilon)
        data_csv[f'sigma_{count}'] = pd.Series(sigma)
        data_csv.to_csv('Stress_strain.csv', index=False, sep=',')

        # ---------------------------------------------------------------------
        # Compute the objective value: relative L2 norm of stress mismatch.
        # ---------------------------------------------------------------------
        # Interpolate the simulated curve onto the same strain grid used for the
        # experimental curve, then normalize the mismatch by the experimental norm.
        f_1 = interp1d(epsilon, sigma, kind='linear')
        sigma_1 = f_1(optimization_range)
        sigma_difference = sigma_1 - sigma_e
        sigma_sqrt = norm(sigma_difference)
        sigma_e_sqrt = norm(sigma_e)
        error = sigma_sqrt / sigma_e_sqrt

        # Store this simulation's error in the global optimization log.
        with open('Error.csv', 'a', newline='') as file:
            writer = csv.writer(file)
            writer.writerow([error])

        # Return values to PySwarms are collected per population.
        error1.append(error)

    # -------------------------------------------------------------------------
    # Optional global stop criterion.
    # -------------------------------------------------------------------------
    # The optimizer normally runs for the requested number of iterations. This
    # check scans all historical errors and stops returning numeric costs once a
    # sufficiently small error has been observed.
    err = pd.read_csv('Error.csv', usecols=['error'])
    err = np.asarray(err).squeeze()
    err = err[~np.isnan(err)]

    min_value = np.min(err)
    if min_value > 0.006:
        return np.array(error1)
    else:
        return print('error achieved')


## 7. PSO Execution

Set the swarm hyperparameters and parameter bounds, then launch the global-best PSO search.


In [ ]:
# -----------------------------------------------------------------------------
# Configure and launch the particle swarm optimization.
# -----------------------------------------------------------------------------
# c1 controls attraction toward each particle's best position, c2 controls
# attraction toward the swarm's global best position, and w is the inertia weight.
options = {'c1': 1.5, 'c2': 1.5, 'w': 0.5}

# Parameter bounds follow the same order used inside cost_function:
#   [h_0_sl-sl, a_sl, xi_inf_sl, xi_0_sl]
# Stress-like quantities are in Pa, matching DAMASK material.yaml units.
max_bound = np.array([3000000000, 2.5, 450000000, 151500000])
min_bound = np.array([500000000, 1, 100000000, 75750000])
bounds = (min_bound, max_bound)

# GlobalBestPSO shares information through one global best particle. Each
# iteration evaluates n_particles DAMASK simulations, so the total maximum number
# of simulations here is n_particles * iters = 20 * 200 = 4000.
optimizer = ps.single.GlobalBestPSO(n_particles=n_particles,
                                    dimensions=dimensions,
                                    options=options,
                                    bounds=bounds)

# Run the optimizer. cost is the best error found by PySwarms, and pos is the
# corresponding optimized parameter vector in the order listed above.
cost, pos = optimizer.optimize(cost_function, iters=200)


2026-05-03 12:13:50,412 - pyswarms.single.global_best - INFO - Optimize for 200 iters with {'c1': 1.5, 'c2': 1.5, 'w': 0.5}
pyswarms.single.global_best:   0%|                                        |0/200